In [7]:
import csv, time, tracemalloc
import pandas as pd
from math import comb
from itertools import combinations
from collections import Counter, defaultdict

PATH = "groceries.csv"
transactions = [frozenset(i.strip() for i in row if i.strip()) for row in csv.reader(open(PATH))]
transactions = [t for t in transactions if t]
items = sorted({i for t in transactions for i in t})
N = len(transactions)
sizes = [len(t) for t in transactions]
item_counts = Counter(i for t in transactions for i in t)

print(f"transactions   {N:,}")
print(f"unique items   {len(items)}")
print(f"items sold     {sum(sizes):,}")
print(f"basket size    min {min(sizes)}   avg {sum(sizes)/N:.2f}   max {max(sizes)}")
print(f"density        {sum(sizes)/(N*len(items)):.2%} of the {N:,} x {len(items)} matrix")
print()
print(f"{'top item':<22}{'count':>8}{'support':>10}")
for it, c in item_counts.most_common(10):
    print(f"{it:<22}{c:>8,}{c/N:>10.2%}")
print()
print(f"{'search space':<22}{'candidates':>18}")
for k in range(1, 6):
    print(f"{'k = '+str(k):<22}{comb(len(items), k):>18,}")
print(f"{'all sizes':<22}{2**len(items)-1:>18.2e}")

transactions   9,835
unique items   169
items sold     43,367
basket size    min 1   avg 4.41   max 32
density        2.61% of the 9,835 x 169 matrix

top item                 count   support
whole milk               2,513    25.55%
other vegetables         1,903    19.35%
rolls/buns               1,809    18.39%
soda                     1,715    17.44%
yogurt                   1,372    13.95%
bottled water            1,087    11.05%
root vegetables          1,072    10.90%
tropical fruit           1,032    10.49%
shopping bags              969     9.85%
sausage                    924     9.40%

search space                  candidates
k = 1                                169
k = 2                             14,196
k = 3                            790,244
k = 4                         32,795,126
k = 5                      1,082,239,158
all sizes                       7.48e+50


In [8]:
MIN_SUPPORT = 0.01
MIN_COUNT = MIN_SUPPORT * N
MAX_K = 2

def brute_force(transactions, items, min_count, max_k):
    stat = {"passes": 0, "candidates": 0, "checks": 0}
    support = {}
    for k in range(1, max_k + 1):
        for c in combinations(items, k):
            cand = frozenset(c)
            stat["candidates"] += 1
            stat["passes"] += 1
            hits = 0
            for t in transactions:
                stat["checks"] += 1
                if cand <= t:
                    hits += 1
            support[cand] = hits
    return {c: s for c, s in support.items() if s >= min_count}, support, stat

def secs(s):
    for unit, step in (("s", 60), ("min", 60), ("hours", 24), ("days", 365)):
        if s < step:
            return f"{s:,.1f} {unit}"
        s /= step
    return f"{s:,.1f} years"

def mem(b):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if b < 1024:
            return f"{b:,.1f} {unit}"
        b /= 1024
    return f"{b:,.1f} PB"

tracemalloc.start()
t0 = time.perf_counter()
bf_frequent, bf_support, bf = brute_force(transactions, items, MIN_COUNT, MAX_K)
bf["time"] = time.perf_counter() - t0
bf["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

per_check = bf["time"] / bf["checks"]
per_cand = bf["peak"] / bf["candidates"]

print(f"BRUTE FORCE   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   k = 1..{MAX_K}")
print(f"  time            {secs(bf['time'])}")
print(f"  peak memory     {mem(bf['peak'])}")
print(f"  data passes     {bf['passes']:,}   1 per candidate")
print(f"  candidates      {bf['candidates']:,}   every itemset of size 1..{MAX_K}")
print(f"  subset checks   {bf['checks']:,}   {bf['checks']/bf['time']/1e6:.1f} M/s, {per_check*1e9:.0f} ns each")
print(f"  frequent        {len(bf_frequent)}   " + "  ".join(f"k={k}: {v}" for k, v in sorted(Counter(len(c) for c in bf_frequent).items())))
print(f"  hit rate        {len(bf_frequent)/bf['candidates']:.2%} of candidates")
print()
print(f"  {'deeper':>6}{'candidates':>16}{'checks':>22}{'time':>14}{'memory':>12}")
for k in range(1, 7):
    c = comb(len(items), k)
    print(f"  {'k = '+str(k):>6}{c:>16,}{c*N:>22,}{secs(c*N*per_check):>14}{mem(c*per_cand):>12}")

bf_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in bf_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

BRUTE FORCE   min_support 1% = 98/9,835   k = 1..2
  time            56.4 s
  peak memory     3.6 MB
  data passes     14,365   1 per candidate
  candidates      14,365   every itemset of size 1..2
  subset checks   141,279,775   2.5 M/s, 399 ns each
  frequent        301   k=1: 88  k=2: 213
  hit rate        2.10% of candidates

  deeper      candidates                checks          time      memory
   k = 1             169             1,662,115         0.7 s     42.8 KB
   k = 2          14,196           139,617,660        55.7 s      3.5 MB
   k = 3         790,244         7,772,049,740      51.7 min    195.5 MB
   k = 4      32,795,126       322,540,064,210      1.5 days      7.9 GB
   k = 5   1,082,239,158    10,643,822,118,930     49.1 days    261.5 GB
   k = 6  29,581,203,652   290,931,137,917,420     3.7 years      7.0 TB


In [9]:
def apriori(transactions, min_count, max_k=None):
    stat = {"passes": 0, "candidates": 0, "checks": 0, "pruned": 0, "per_level": []}
    counts = defaultdict(int)
    for t in transactions:
        for i in t:
            counts[frozenset([i])] += 1
            stat["checks"] += 1
    stat["passes"] += 1
    stat["candidates"] += len(counts)
    level = {c: s for c, s in counts.items() if s >= min_count}
    frequent = dict(level)
    stat["per_level"].append((1, len(counts), len(level)))
    k = 1
    while level and (max_k is None or k < max_k):
        k += 1
        prev = sorted(level)
        cands = set()
        for a, b in combinations(prev, 2):
            u = a | b
            if len(u) != k or u in cands:
                continue
            if all(frozenset(s) in level for s in combinations(sorted(u), k - 1)):
                cands.add(u)
            else:
                stat["pruned"] += 1
        stat["candidates"] += len(cands)
        if not cands:
            stat["per_level"].append((k, 0, 0))
            break
        counts = defaultdict(int)
        for t in transactions:
            if len(t) >= k:
                for s in combinations(sorted(t), k):
                    stat["checks"] += 1
                    fs = frozenset(s)
                    if fs in cands:
                        counts[fs] += 1
        stat["passes"] += 1
        level = {c: s for c, s in counts.items() if s >= min_count}
        frequent.update(level)
        stat["per_level"].append((k, len(cands), len(level)))
    return frequent, stat

tracemalloc.start()
t0 = time.perf_counter()
ap_frequent, ap = apriori(transactions, MIN_COUNT)
ap["time"] = time.perf_counter() - t0
ap["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"APRIORI   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   k unlimited")
print(f"  time            {secs(ap['time'])}")
print(f"  peak memory     {mem(ap['peak'])}")
print(f"  data passes     {ap['passes']}   1 per size")
print(f"  candidates      {ap['candidates']:,}   {ap['pruned']:,} pruned before counting")
print(f"  subset checks   {ap['checks']:,}   {ap['checks']/ap['time']/1e6:.1f} M/s")
print(f"  frequent        {len(ap_frequent)}   up to k={max(len(c) for c in ap_frequent)}")
print()
print(f"  {'level':>6}{'candidates':>14}{'frequent':>12}{'survival':>11}")
for k, c, f in ap["per_level"]:
    print(f"  {'k = '+str(k):>6}{c:>14,}{f:>12,}{(f/c if c else 0):>11.1%}")

ap_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in ap_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

APRIORI   min_support 1% = 98/9,835   k unlimited
  time            5.6 s
  peak memory     1.9 MB
  data passes     4   1 per size
  candidates      4,579   2,499 pruned before counting
  subset checks   1,678,796   0.3 M/s
  frequent        333   up to k=3

   level    candidates    frequent   survival
   k = 1           169          88      52.1%
   k = 2         3,828         213       5.6%
   k = 3           576          32       5.6%
   k = 4             6           0       0.0%


In [10]:
from array import array

BUCKETS = 2000
ITEM_ID = {it: i for i, it in enumerate(items)}
H1 = lambda a, b: (a * len(items) + b) % BUCKETS

def pcy(transactions, min_count, n_buckets):
    stat = {"passes": 0, "candidates": 0, "checks": 0, "hashed": 0}
    counts = defaultdict(int)
    buckets = array("i", bytes(4 * n_buckets))
    for t in transactions:
        ids = sorted(ITEM_ID[i] for i in t)
        for i in t:
            counts[i] += 1
        for a, b in combinations(ids, 2):
            buckets[H1(a, b)] += 1
            stat["hashed"] += 1
    stat["passes"] += 1
    freq1 = {i for i, c in counts.items() if c >= min_count}
    bitmap = bytearray(1 if c >= min_count else 0 for c in buckets)
    stat["table_bytes"] = 4 * n_buckets
    stat["bitmap_bytes"] = n_buckets
    stat["frequent_buckets"] = sum(bitmap)
    stat["occurring"] = 0
    pairs = defaultdict(int)
    seen = set()
    for t in transactions:
        for a, b in combinations(sorted(i for i in t if i in freq1), 2):
            stat["checks"] += 1
            seen.add((a, b))
            if bitmap[H1(ITEM_ID[a], ITEM_ID[b])]:
                pairs[frozenset((a, b))] += 1
    stat["passes"] += 1
    stat["occurring"] = len(seen)
    stat["candidates"] = len(freq1) + len(pairs)
    frequent = {frozenset([i]): counts[i] for i in freq1}
    frequent.update({p: c for p, c in pairs.items() if c >= min_count})
    return frequent, freq1, stat

tracemalloc.start()
t0 = time.perf_counter()
pcy_frequent, pcy_f1, pcy_s = pcy(transactions, MIN_COUNT, BUCKETS)
pcy_s["time"] = time.perf_counter() - t0
pcy_s["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"PCY   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   pairs only, {BUCKETS:,} buckets")
print(f"  time            {secs(pcy_s['time'])}")
print(f"  peak memory     {mem(pcy_s['peak'])}")
print(f"  data passes     2   count+hash, then count surviving pairs")
print(f"  pairs hashed    {pcy_s['hashed']:,}   into {BUCKETS:,} buckets")
print(f"  hash table      {mem(pcy_s['table_bytes'])} of 4-byte counters -> {mem(pcy_s['bitmap_bytes'])} as a bitmap")
print(f"  frequent bkts   {pcy_s['frequent_buckets']:,}   {pcy_s['frequent_buckets']/BUCKETS:.1%} of buckets")
print(f"  pair funnel     {comb(len(pcy_f1),2):,} from frequent items -> {pcy_s['occurring']:,} ever co-occur -> {pcy_s['candidates']-len(pcy_f1):,} past the bitmap")
print(f"  candidates      {pcy_s['candidates']:,}   {len(pcy_f1)} items + {pcy_s['candidates']-len(pcy_f1):,} pairs")
print(f"  frequent        {len(pcy_frequent)}   k=1: {len(pcy_f1)}  k=2: {len(pcy_frequent)-len(pcy_f1)}")
print()
print(f"  {'buckets':>9}{'table':>10}{'bitmap':>10}{'frequent':>10}{'candidates':>12}")
occurring = sorted({tuple(sorted(ITEM_ID[i] for i in p)) for t in transactions for p in combinations([i for i in t if i in pcy_f1], 2)})
for nb in (200, 2000, 20000, 200000):
    b = array("i", bytes(4 * nb))
    for t in transactions:
        for a, c in combinations(sorted(ITEM_ID[i] for i in t), 2):
            b[(a * len(items) + c) % nb] += 1
    bm = bytearray(1 if x >= MIN_COUNT else 0 for x in b)
    print(f"  {nb:>9,}{mem(4*nb):>10}{mem(nb):>10}{sum(bm)/nb:>10.1%}{sum(1 for a, c in occurring if bm[(a*len(items)+c) % nb]):>12,}")

pcy_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in pcy_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

PCY   min_support 1% = 98/9,835   pairs only, 2,000 buckets
  time            0.7 s
  peak memory     588.2 KB
  data passes     2   count+hash, then count surviving pairs
  pairs hashed    137,278   into 2,000 buckets
  hash table      7.8 KB of 4-byte counters -> 2.0 KB as a bitmap
  frequent bkts   401   20.1% of buckets
  pair funnel     3,828 from frequent items -> 3,781 ever co-occur -> 1,214 past the bitmap
  candidates      1,302   88 items + 1,214 pairs
  frequent        301   k=1: 88  k=2: 213

    buckets     table    bitmap  frequent  candidates
        200   800.0 B   200.0 B    100.0%       3,781
      2,000    7.8 KB    2.0 KB     20.1%       1,214
     20,000   78.1 KB   19.5 KB      1.1%         246
    200,000  781.2 KB  195.3 KB      0.1%         213


In [11]:
BUCKETS1 = 2000
BUCKETS2 = 1999

def multistage(transactions, min_count, nb1, nb2):
    stat = {"passes": 0, "candidates": 0, "checks": 0, "hashed1": 0, "hashed2": 0}
    counts = defaultdict(int)
    t1 = array("i", bytes(4 * nb1))
    for t in transactions:
        for i in t:
            counts[i] += 1
        for a, b in combinations(sorted(ITEM_ID[i] for i in t), 2):
            t1[(a * len(items) + b) % nb1] += 1
            stat["hashed1"] += 1
    stat["passes"] += 1
    freq1 = {i for i, c in counts.items() if c >= min_count}
    bm1 = bytearray(1 if c >= min_count else 0 for c in t1)
    del t1
    t2 = array("i", bytes(4 * nb2))
    stage1 = set()
    for t in transactions:
        for a, b in combinations(sorted(ITEM_ID[i] for i in t if i in freq1), 2):
            stat["checks"] += 1
            key = a * len(items) + b
            if bm1[key % nb1]:
                t2[(key * 7919) % nb2] += 1
                stat["hashed2"] += 1
                stage1.add((a, b))
    stat["passes"] += 1
    bm2 = bytearray(1 if c >= min_count else 0 for c in t2)
    del t2
    stat["bitmap_bytes"] = nb1 + nb2
    stat["frequent1"] = sum(bm1)
    stat["frequent2"] = sum(bm2)
    stat["after1"] = len(stage1)
    pairs = defaultdict(int)
    for t in transactions:
        for a, b in combinations(sorted(i for i in t if i in freq1), 2):
            stat["checks"] += 1
            key = ITEM_ID[a] * len(items) + ITEM_ID[b]
            if bm1[key % nb1] and bm2[(key * 7919) % nb2]:
                pairs[frozenset((a, b))] += 1
    stat["passes"] += 1
    stat["candidates"] = len(freq1) + len(pairs)
    frequent = {frozenset([i]): counts[i] for i in freq1}
    frequent.update({p: c for p, c in pairs.items() if c >= min_count})
    return frequent, freq1, stat

tracemalloc.start()
t0 = time.perf_counter()
ms_frequent, ms_f1, ms_s = multistage(transactions, MIN_COUNT, BUCKETS1, BUCKETS2)
ms_s["time"] = time.perf_counter() - t0
ms_s["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"MULTISTAGE   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   pairs only, buckets {BUCKETS1:,} + {BUCKETS2:,}")
print(f"  time            {secs(ms_s['time'])}")
print(f"  peak memory     {mem(ms_s['peak'])}")
print(f"  data passes     3   count+hash1, rehash1, count survivors")
print(f"  pairs hashed    {ms_s['hashed1']:,} into table 1   {ms_s['hashed2']:,} into table 2")
print(f"  bitmaps kept    {mem(ms_s['bitmap_bytes'])} for both stages, never held with a counter table")
print(f"  frequent bkts   table 1 {ms_s['frequent1']:,} ({ms_s['frequent1']/BUCKETS1:.1%})   table 2 {ms_s['frequent2']:,} ({ms_s['frequent2']/BUCKETS2:.1%})")
print(f"  pair funnel     {comb(len(ms_f1),2):,} from frequent items -> {ms_s['after1']:,} past bitmap 1 -> {ms_s['candidates']-len(ms_f1):,} past both")
print(f"  candidates      {ms_s['candidates']:,}   {len(ms_f1)} items + {ms_s['candidates']-len(ms_f1):,} pairs, {len(ms_frequent)-len(ms_f1)} of them real")
print(f"  frequent        {len(ms_frequent)}   k=1: {len(ms_f1)}  k=2: {len(ms_frequent)-len(ms_f1)}")

ms_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in ms_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

MULTISTAGE   min_support 1% = 98/9,835   pairs only, buckets 2,000 + 1,999
  time            1.8 s
  peak memory     200.4 KB
  data passes     3   count+hash1, rehash1, count survivors
  pairs hashed    137,278 into table 1   70,787 into table 2
  bitmaps kept    3.9 KB for both stages, never held with a counter table
  frequent bkts   table 1 401 (20.1%)   table 2 251 (12.6%)
  pair funnel     3,828 from frequent items -> 1,214 past bitmap 1 -> 418 past both
  candidates      506   88 items + 418 pairs, 213 of them real
  frequent        301   k=1: 88  k=2: 213


In [12]:
CHUNKS = 10

def son(transactions, min_support, n_chunks):
    stat = {"passes": 0, "checks": 0, "local_passes": 0, "chunks": []}
    size = -(-len(transactions) // n_chunks)
    parts = [transactions[i:i + size] for i in range(0, len(transactions), size)]
    candidates = set()
    for ch in parts:
        thr = min_support * len(ch)
        local, st = apriori(ch, thr)
        stat["checks"] += st["checks"]
        stat["local_passes"] += st["passes"]
        stat["chunks"].append((len(ch), thr, len(local), max(len(c) for c in local), len(set(local) - candidates)))
        candidates |= set(local)
    stat["passes"] += 1
    stat["candidates"] = len(candidates)
    by_size = defaultdict(set)
    for c in candidates:
        by_size[len(c)].add(c)
    counts = defaultdict(int)
    for t in transactions:
        for k, group in by_size.items():
            if len(t) >= k:
                for s in combinations(sorted(t), k):
                    stat["checks"] += 1
                    fs = frozenset(s)
                    if fs in group:
                        counts[fs] += 1
    stat["passes"] += 1
    frequent = {c: v for c, v in counts.items() if v >= min_support * len(transactions)}
    return frequent, stat

tracemalloc.start()
t0 = time.perf_counter()
son_frequent, son_s = son(transactions, MIN_SUPPORT, CHUNKS)
son_s["time"] = time.perf_counter() - t0
son_s["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"SON   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   {CHUNKS} chunks, local threshold {MIN_SUPPORT:.0%} of each chunk")
print(f"  time            {secs(son_s['time'])}")
print(f"  peak memory     {mem(son_s['peak'])}")
print(f"  data passes     2   map/reduce 1 mines chunks, map/reduce 2 counts the union")
print(f"  chunk passes    {son_s['local_passes']}   in memory, one chunk at a time")
print(f"  subset checks   {son_s['checks']:,}")
print(f"  candidates      {son_s['candidates']:,}   union of local frequent sets, {son_s['candidates']-len(son_frequent):,} turn out to be false positives")
print(f"  frequent        {len(son_frequent)}   " + "  ".join(f"k={k}: {v}" for k, v in sorted(Counter(len(c) for c in son_frequent).items())))
print()
print(f"  {'chunk':>6}{'rows':>7}{'threshold':>11}{'local freq':>12}{'max k':>7}{'new to union':>14}")
for n, (rows, thr, loc, mk, new) in enumerate(son_s["chunks"], 1):
    print(f"  {n:>6}{rows:>7,}{thr:>11.1f}{loc:>12,}{mk:>7}{new:>14,}")

son_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in son_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

SON   min_support 1% = 98/9,835   10 chunks, local threshold 1% of each chunk
  time            12.5 s
  peak memory     2.3 MB
  data passes     2   map/reduce 1 mines chunks, map/reduce 2 counts the union
  chunk passes    40   in memory, one chunk at a time
  subset checks   3,357,592
  candidates      925   union of local frequent sets, 592 turn out to be false positives
  frequent        333   k=1: 88  k=2: 213  k=3: 32

   chunk   rows  threshold  local freq  max k  new to union
       1    984        9.8         340      3           340
       2    984        9.8         580      4           295
       3    984        9.8         334      3            34
       4    984        9.8         420      4            60
       5    984        9.8         562      4           108
       6    984        9.8         382      4            24
       7    984        9.8         362      3            14
       8    984        9.8         290      3             6
       9    984        9.8    

In [13]:
import random

FRACTION = 0.20
LOWER = 0.8
DECAY = 0.8
MAX_ATTEMPTS = 6

def negative_border(frequent, universe):
    by_size = defaultdict(set)
    for c in frequent:
        by_size[len(c)].add(c)
    border = {frozenset([i]) for i in universe} - by_size[1]
    k = 1
    while by_size[k]:
        k += 1
        prev = by_size[k - 1]
        for a, b in combinations(sorted(prev), 2):
            u = a | b
            if len(u) == k and u not in by_size[k] and all(frozenset(s) in prev for s in combinations(sorted(u), k - 1)):
                border.add(u)
    return border

def toivonen(transactions, min_count, fraction, lower, decay, universe, max_attempts, seed=0):
    rnd = random.Random(seed)
    stat = {"passes": 0, "sample_passes": 0, "checks": 0, "attempts": [], "candidates": 0}
    for _ in range(max_attempts):
        sample = rnd.sample(transactions, int(fraction * len(transactions)))
        thr = lower * (min_count / len(transactions)) * len(sample)
        s_freq, st = apriori(sample, thr)
        stat["checks"] += st["checks"]
        stat["sample_passes"] += st["passes"]
        border = negative_border(s_freq, universe)
        check = defaultdict(set)
        for c in set(s_freq) | border:
            check[len(c)].add(c)
        counts = defaultdict(int)
        for t in transactions:
            for k, group in check.items():
                if len(t) >= k:
                    for s in combinations(sorted(t), k):
                        stat["checks"] += 1
                        fs = frozenset(s)
                        if fs in group:
                            counts[fs] += 1
        stat["passes"] += 1
        bad = sum(1 for c in border if counts[c] >= min_count)
        stat["attempts"].append((lower, len(sample), thr, len(s_freq), len(border), bad))
        if not bad:
            stat["candidates"] = len(s_freq) + len(border)
            return {c: counts[c] for c in s_freq if counts[c] >= min_count}, stat
        lower *= decay
    return None, stat

tracemalloc.start()
t0 = time.perf_counter()
tv_frequent, tv_s = toivonen(transactions, MIN_COUNT, FRACTION, LOWER, DECAY, items, MAX_ATTEMPTS)
tv_s["time"] = time.perf_counter() - t0
tv_s["peak"] = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"TOIVONEN   min_support {MIN_SUPPORT:.0%} = {MIN_COUNT:.0f}/{N:,}   sample {FRACTION:.0%}, sample threshold from {LOWER:.0%}, x{DECAY} per restart")
print(f"  time            {secs(tv_s['time'])}")
print(f"  peak memory     {mem(tv_s['peak'])}")
print(f"  attempts        {len(tv_s['attempts'])}   a restart costs one more full pass")
print(f"  data passes     {tv_s['passes']} full, {len(tv_s['attempts'])} over the {FRACTION:.0%} sample ({tv_s['sample_passes']} in-memory sample passes)")
print(f"  subset checks   {tv_s['checks']:,}")
print(f"  candidates      {tv_s['candidates']:,}   sample frequent + negative border, counted together in the accepted pass")
print(f"  frequent        {len(tv_frequent)}   " + "  ".join(f"k={k}: {v}" for k, v in sorted(Counter(len(c) for c in tv_frequent).items())))
print()
print(f"  {'attempt':>8}{'factor':>8}{'sample':>8}{'threshold':>11}{'sample freq':>13}{'border':>9}{'border hits':>13}{'verdict':>10}")
for n, (low, rows, thr, sf, nb, bad) in enumerate(tv_s["attempts"], 1):
    print(f"  {n:>8}{low:>8.2f}{rows:>8,}{thr:>11.1f}{sf:>13,}{nb:>9,}{bad:>13,}{('restart' if bad else 'accept'):>10}")

tv_df = pd.DataFrame(
    [{"itemset": tuple(sorted(c)), "size": len(c), "count": s, "support": s / N} for c, s in tv_frequent.items()]
).sort_values(["size", "count"], ascending=[True, False]).reset_index(drop=True)

TOIVONEN   min_support 1% = 98/9,835   sample 20%, sample threshold from 80%, x0.8 per restart
  time            11.4 s
  peak memory     6.0 MB
  attempts        2   a restart costs one more full pass
  data passes     2 full, 2 over the 20% sample (8 in-memory sample passes)
  subset checks   4,006,960
  candidates      7,789   sample frequent + negative border, counted together in the accepted pass
  frequent        333   k=1: 88  k=2: 213  k=3: 32

   attempt  factor  sample  threshold  sample freq   border  border hits   verdict
         1    0.80   1,967       15.7          505    5,308            7   restart
         2    0.64   1,967       12.6          764    7,025            0    accept
